<h3><b>Stochastic Gradient Descent</b></h3>

In [ ]:

#    (Batch) Gradient Descent:

X = data_input
Y = labels
m = X.shape[1]  # Number of training examples
parameters = initialize_parameters(layers_dims)
for i in range(0, num_iterations):
    # Forward propagation
    a, caches = forward_propagation(X, parameters)
    # Compute cost
    cost_total = compute_cost(a, Y)  # Cost for m training examples
    # Backward propagation
    grads = backward_propagation(a, caches, parameters)
    # Update parameters
    parameters = update_parameters(parameters, grads)
    # Compute average cost
    cost_avg = cost_total / m

#    Stochastic Gradient Descent:
"""
implementing SGD requires 3 for-loops in total:

   1. Over the number of iterations
   2. Over the 𝑚 training examples
   3. Over the layers (to update all parameters, from (𝑊[1],𝑏[1]) to (𝑊[𝐿],𝑏[𝐿]))
"""

X = data_input
Y = labels
m = X.shape[1]  # Number of training examples
parameters = initialize_parameters(layers_dims)
for i in range(0, num_iterations):
    cost_total = 0
    for j in range(0, m):
        # Forward propagation
        a, caches = forward_propagation(X[:,j], parameters)
        # Compute cost
        cost_total += compute_cost(a, Y[:,j])  # Cost for one training example
        # Backward propagation
        grads = backward_propagation(a, caches, parameters)
        # Update parameters
        parameters = update_parameters(parameters, grads)
    # Compute average cost
    cost_avg = cost_total / m

<h3><b> Mini-Batch Gradient Descent</b></h3>

In [ ]:
"""
There are two steps in implementing mini-batch gradient descent on dataset (X, Y):

    1. Shuffle: Create a shuffled version of the training set (X, Y)
    2. Partition: Partition the shuffled (X, Y) into mini-batches of size mini_batch_size (commen mini-batch sizes:16, 32, 64, 128, 256, 512, powers of two),
       the last mini batch might be smaller.
"""

def random_mini_batches(X, Y, mini_batch_size = 64, seed = 0):
    """
    Creates a list of random minibatches from (X, Y)
    
    Arguments:
    X -- input data, of shape (input size, number of examples)
    Y -- true "label" vector (1 for blue dot / 0 for red dot), of shape (1, number of examples)
    mini_batch_size -- size of the mini-batches, integer
    
    Returns:
    mini_batches -- list of synchronous (mini_batch_X, mini_batch_Y)
    """
    
    np.random.seed(seed)            
    m = X.shape[1]                  
    mini_batches = []
        
    # Step 1: Shuffle (X, Y)
    permutation = list(np.random.permutation(m))
    shuffled_X = X[:, permutation]
    shuffled_Y = Y[:, permutation].reshape((1, m))
    
    inc = mini_batch_size

    # Step 2 - Partition (shuffled_X, shuffled_Y).
    
    num_complete_minibatches = math.floor(m / mini_batch_size) 
    for k in range(0, num_complete_minibatches):
      
        mini_batch_X = shuffled_X[:, k*inc : k*inc+inc]
        mini_batch_Y = shuffled_Y[:, k*inc : k*inc+inc]
       
        mini_batch = (mini_batch_X, mini_batch_Y)
        mini_batches.append(mini_batch)
    
    # For handling the end case (last mini-batch < mini_batch_size i.e less than 64)
    if m % mini_batch_size != 0:
       
        mini_batch_X = shuffled_X[:, num_complete_minibatches*inc:]
        mini_batch_Y = shuffled_Y[:, num_complete_minibatches*inc:]
       
        mini_batch = (mini_batch_X, mini_batch_Y)
        mini_batches.append(mini_batch)
    
    return mini_batches

<h3><b>Momentum</b></h3>

In [ ]:
"""
- The velocity is initialized with zeros. So the algorithm will take a few iterations to "build up" velocity and start to take bigger steps.

- If 𝛽=0 then this just becomes standard gradient descent without momentum
"""

def initialize_velocity(parameters):
    """
    Initializes the velocity as a python dictionary with:
                - keys: "dW1", "db1", ..., "dWL", "dbL" 
                - values: numpy arrays of zeros of the same shape as the corresponding gradients/parameters.
    Arguments:
    parameters -- python dictionary containing your parameters.
                    parameters['W' + str(l)] = Wl
                    parameters['b' + str(l)] = bl
    
    Returns:
    v -- python dictionary containing the current velocity.
                    v['dW' + str(l)] = velocity of dWl
                    v['db' + str(l)] = velocity of dbl
    """
    L = len(parameters) // 2 
    v = {}
    
    for l in range(1, L + 1):
       
        v[f"dW{l}"] = np.zeros(parameters[f"W{l}"].shape)
        v[f"db{l}"] = np.zeros(parameters[f"b{l}"].shape)
        
    return v
    

def update_parameters_with_momentum(parameters, grads, v, beta, learning_rate):
    """
    Update parameters using Momentum
    
    Arguments:
    parameters -- python dictionary containing your parameters:
                    parameters['W' + str(l)] = Wl
                    parameters['b' + str(l)] = bl
    grads -- python dictionary containing your gradients for each parameters:
                    grads['dW' + str(l)] = dWl
                    grads['db' + str(l)] = dbl
    v -- python dictionary containing the current velocity:
                    v['dW' + str(l)] = ...
                    v['db' + str(l)] = ...
    beta -- the momentum hyperparameter, scalar
    learning_rate -- the learning rate, scalar
    
    Returns:
    parameters -- python dictionary containing your updated parameters 
    v -- python dictionary containing your updated velocities
    """

    L = len(parameters) // 2 
    
    for l in range(1, L + 1):
        

        v[f"dW{l}"] = beta * v[f"dW{l}"] + (1 - beta) * grads[f"dW{l}"]
        v[f"db{l}"] = beta * v[f"db{l}"] + (1 - beta) * grads[f"db{l}"]
        parameters[f"W{l}"] -= learning_rate * v[f"dW{l}"]
        parameters[f"b{l}"] -= learning_rate * v[f"db{l}"]
        
    return parameters, v

<h3><b>Adam Algorithm</b></h3>

In [ ]:
def initialize_adam(parameters) :
    """
    Initializes v and s as two python dictionaries with:
                - keys: "dW1", "db1", ..., "dWL", "dbL" 
                - values: numpy arrays of zeros of the same shape as the corresponding gradients/parameters.
    
    Arguments:
    parameters -- python dictionary containing your parameters.
                    parameters["W" + str(l)] = Wl
                    parameters["b" + str(l)] = bl
    
    Returns: 
    v -- python dictionary that will contain the exponentially weighted average of the gradient. Initialized with zeros.
                    v["dW" + str(l)] = ...
                    v["db" + str(l)] = ...
    s -- python dictionary that will contain the exponentially weighted average of the squared gradient. Initialized with zeros.
                    s["dW" + str(l)] = ...
                    s["db" + str(l)] = ...

    """
    
    L = len(parameters) // 2 
    v = {}
    s = {}
    
    
    for l in range(1, L + 1):
    
        v[f"dW{l}"] = np.zeros(parameters[f"W{l}"].shape)
        v[f"db{l}"] = np.zeros(parameters[f"b{l}"].shape)
        s[f"dW{l}"] = np.zeros(parameters[f"W{l}"].shape)
        s[f"db{l}"] = np.zeros(parameters[f"b{l}"].shape)
    
    return v, s
    

def update_parameters_with_adam(parameters, grads, v, s, t, learning_rate = 0.01, beta1 = 0.9, beta2 = 0.999,  epsilon = 1e-8):
    """
    Update parameters using Adam
    
    Arguments:
    parameters -- python dictionary containing your parameters:
                    parameters['W' + str(l)] = Wl
                    parameters['b' + str(l)] = bl
    grads -- python dictionary containing your gradients for each parameters:
                    grads['dW' + str(l)] = dWl
                    grads['db' + str(l)] = dbl
    v -- Adam variable, moving average of the first gradient, python dictionary
    s -- Adam variable, moving average of the squared gradient, python dictionary
    t -- Adam variable, counts the number of taken steps
    learning_rate -- the learning rate, scalar.
    beta1 -- Exponential decay hyperparameter for the first moment estimates 
    beta2 -- Exponential decay hyperparameter for the second moment estimates 
    epsilon -- hyperparameter preventing division by zero in Adam updates

    Returns:
    parameters -- python dictionary containing your updated parameters 
    v -- Adam variable, moving average of the first gradient, python dictionary
    s -- Adam variable, moving average of the squared gradient, python dictionary
    """
    
    L = len(parameters) // 2                
    v_corrected = {}                         
    s_corrected = {}                         
    
    
    for l in range(1, L + 1):
        
        # momentum terms  
        v[f"dW{l}"] = beta1 * v[f"dW{l}"] + ((1 - beta1) * grads[f"dW{l}"])
        v[f"db{l}"] = beta1 * v[f"db{l}"] + ((1 - beta1) * grads[f"db{l}"])

        # bias correction of momentum terms
        v_corrected[f"dW{l}"] = v[f"dW{l}"] / (1 - np.power(beta1, t))
        v_corrected[f"db{l}"] = v[f"db{l}"] / (1 - np.power(beta1, t))

        # RMSprop terms
        s[f"dW{l}"] = beta2 * s[f"dW{l}"] + ((1 - beta2) * np.power(grads[f"dW{l}"], 2))
        s[f"db{l}"] = beta2 * s[f"db{l}"] + ((1 - beta2) * np.power(grads[f"db{l}"], 2))

        # bias correction of RMSprop terms
        s_corrected[f"dW{l}"] = s[f"dW{l}"] / (1 - np.power(beta2, t))
        s_corrected[f"db{l}"] = s[f"db{l}"] / (1 - np.power(beta2, t))
        
        # update parameters
        parameters[f"W{l}"] -= learning_rate * (v_corrected[f"dW{l}"] / np.sqrt(s_corrected[f"dW{l}"]+epsilon))
        parameters[f"b{l}"] -= learning_rate * (v_corrected[f"db{l}"] / np.sqrt(s_corrected[f"db{l}"]+epsilon))
         

    return parameters, v, s, v_corrected, s_corrected

<h3><b>Learning Rate Decay</b></h3>

In [ ]:
# Calculate the new learning rate using exponential weight decay
def update_lr(learning_rate0, epoch_num, decay_rate):
    """
    Calculates updated the learning rate using exponential weight decay.
    
    Arguments:
    learning_rate0 -- Original learning rate. Scalar
    epoch_num -- Epoch number. Integer
    decay_rate -- Decay rate. Scalar

    Returns:
    learning_rate -- Updated learning rate. Scalar 
    """
    learning_rate = 1 / (1 + decay_rate * epoch_num) * learning_rate0
    
    return learning_rate

# Fixed Interval Scheduling
def schedule_lr_decay(learning_rate0, epoch_num, decay_rate, time_interval=1000):
    """
    Calculates updated the learning rate using exponential weight decay.
    
    Arguments:
    learning_rate0 -- Original learning rate. Scalar
    epoch_num -- Epoch number. Integer.
    decay_rate -- Decay rate. Scalar.
    time_interval -- Number of epochs where you update the learning rate.

    Returns:
    learning_rate -- Updated learning rate. Scalar 
    """
    learning_rate = (1 * learning_rate0) / (1 + decay_rate * (np.floor(epoch_num/time_interval)))
    
    return learning_rate
    